In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

In [20]:
df = pd.read_csv('mes_donnees_normalisées.csv', sep=';')
df.columns = df.columns.str.strip()
#df = df.drop(columns = ['P1_reel'])
df.loc[df['Lt'] == 'R', 'Lt'] = 0
df.loc[df['Lt'] == 'G', 'Lt'] = 1

In [25]:
#Préparation des données 

# Séparation des features et de la target
df_ut = df[['P1_reel','P2','Lt','Lq_reel','P4','P5','V','E2','P6_reel','E1']]
df_ut = df_ut.dropna()  # Supprimer les lignes avec des valeurs manquantes
X1 = df_ut.drop(columns=['E1'])
y1 = df_ut['E1']

# 3. GESTION DES NaN (Le pansement propre avant le modèle)
# Un Random Forest de base dans scikit-learn ne tolère AUCUN NaN.
# Ici, on choisit d'imputer (remplacer) les valeurs manquantes par la médiane de la colonne.
imputer = SimpleImputer(strategy='median')
# On utilise fit_transform mais on le remet IMMÉDIATEMENT dans un DataFrame pour garder les index et noms de colonnes
X_clean = pd.DataFrame(imputer.fit_transform(X1), columns=X1.columns, index=X1.index)


X_train, X_test, y_train, y_test = train_test_split(X1, y1, test_size=0.2, random_state=42)
"""
#Entraineemnt = les 1ères expériences, Test = les dernières expériences
X1_train = X1.iloc[:500]
y1_train = y1.iloc[:500]

X1_test = X1.iloc[500:703]
y1_test = y1.iloc[500:703]
"""
#Entrainement = 3 1ères données de chaque expérience, Test = les 2 dernières données de chaque expérience






'\n#Entraineemnt = les 1ères expériences, Test = les dernières expériences\nX1_train = X1.iloc[:500]\ny1_train = y1.iloc[:500]\n\nX1_test = X1.iloc[500:703]\ny1_test = y1.iloc[500:703]\n'

In [27]:
# Modèle Random Forest
rf = RandomForestRegressor(
    n_estimators=200,   # nombre d'arbres
    max_depth= 8,    # profondeur illimitée
    random_state=42,
    n_jobs=-1          # utilise tous les cœurs CPU
)

# Entraînement
rf.fit(X_train, y_train)

# Prédictions
y_pred = rf.predict(X_test)
y_pred = pd.Series(y_pred, index=y_test.index)

# Ajout dans results
#results["RF"] = y_pred

# Métriques
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"R2: {r2:.06f}")
print(f"MSE: {mse:.06f}")
print()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse) # La racine carrée pour retrouver l'unité physique de E1

print(f"Erreur Absolue Moyenne (MAE) : {mae:.2f}")
print(f"Racine de l'Erreur Quadratique Moyenne (RMSE) : {rmse:.2f} ")
print("\n")

# Exemple de sauvegarde dans un dataframe de résultats
results = pd.DataFrame({'Vraie_Valeur': y_test, 'Prediction_RF': y_pred})
print("Aperçu des prédictions :")
print(results.head())
print("\n")

print("--- 4. Importance des Paramètres ---")
# On récupère le classement d'importance des variables calculé par le modèle
importances = rf.feature_importances_
for col, imp in sorted(zip(X1.columns, importances), key=lambda x: x[1], reverse=True):
    print(f"{col:<10}: {imp*100:.1f}%")





R2: 0.945763
MSE: 0.001195

Erreur Absolue Moyenne (MAE) : 0.03
Racine de l'Erreur Quadratique Moyenne (RMSE) : 0.03 


Aperçu des prédictions :
     Vraie_Valeur  Prediction_RF
696      0.487565       0.479692
259      0.649460       0.681253
379      0.842328       0.835141
206      0.864852       0.862850
595      0.646645       0.674899


--- 4. Importance des Paramètres ---
P2        : 53.7%
P1_reel   : 17.8%
P5        : 10.3%
E2        : 6.3%
P6_reel   : 4.8%
P4        : 3.9%
Lq_reel   : 3.1%
Lt        : 0.0%
V         : 0.0%
